# Step 7 — Final SHAP Explainability

## Synchronization with the other notebooks

- **Step 5:** model comparison is performed on the validation set; XGBoost is the selected model family.
- **Step 6:** the official XGBoost model is trained, the cost-optimal threshold is selected on validation data, the threshold is frozen, and the final held-out test set is evaluated.
- **Step 4:** the final model and final risk configuration are loaded for transaction-level scoring.
- **Step 7:** this notebook loads those same final artifacts and explains predictions made by the final XGBoost model.

### This notebook does NOT:
- train another model
- create another train/validation/test split
- optimize a threshold
- evaluate the final test set
- create a different model or threshold

The purpose here is only to explain model decisions using SHAP.


In [ ]:
# Step 1: Imports

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

from pathlib import Path

print("Imports completed successfully.")


In [ ]:
# Step 2: Load Final Model

model_dir = Path("../models")

final_model_path = model_dir / "final_xgb_model.pkl"

if not final_model_path.exists():
    raise FileNotFoundError(
        f"Final model not found: {final_model_path.resolve()}\n"
        "Run Step 6 final artifact-saving cell first."
    )

xgb_model = joblib.load(final_model_path)

print("Final XGBoost model loaded successfully.")
print("Model path:", final_model_path.resolve())


In [ ]:
# Step 3: Load Final Risk Configuration

final_config_path = model_dir / "final_risk_config.pkl"

if not final_config_path.exists():
    raise FileNotFoundError(
        f"Final risk configuration not found: {final_config_path.resolve()}\n"
        "Run Step 6 final artifact-saving cell first."
    )

final_config = joblib.load(final_config_path)

FINAL_THRESHOLD = float(final_config["threshold"])

print("Final risk configuration loaded successfully.")
print(final_config)
print(f"\nFrozen decision threshold: {FINAL_THRESHOLD:.2f}")


Final risk configuration loaded successfully.
{'model': 'XGBoost', 'threshold': 0.03, 'false_positive_cost': 100.0, 'false_negative_cost': 5000.0}

Frozen decision threshold: 0.03


In [ ]:
# Step 4: Load Dataset

data_path = Path("../data/creditcard.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)


Dataset shape: (284807, 31)


In [ ]:
#  Step 5: Define Feature Columns

feature_columns = (
    df.drop(columns=["Class"])
      .columns
      .tolist()
)

print("Number of model features:", len(feature_columns))
print(feature_columns)

# Check that the final model expects the same number of features.
if hasattr(xgb_model, "n_features_in_"):
    if xgb_model.n_features_in_ != len(feature_columns):
        raise ValueError(
            f"Feature mismatch: model expects "
            f"{xgb_model.n_features_in_}, but dataset provides "
            f"{len(feature_columns)}."
        )

print("\nFeature/model compatibility check passed.")


Number of model features: 30
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Feature/model compatibility check passed.


In [ ]:
#  Step 6: Create SHAP Explainer

explainer = shap.TreeExplainer(xgb_model)

print("SHAP TreeExplainer created successfully.")


SHAP TreeExplainer created successfully.


## Step 7 — Helper for extracting SHAP values

XGBoost/SHAP versions can return slightly different output shapes.  
This helper normalizes the output so the rest of the notebook works consistently.


In [ ]:
#  Step 7: SHAP Value Helper

def get_shap_row(explainer, transaction):
    """Return one 1-D SHAP vector for a single transaction."""

    shap_output = explainer.shap_values(transaction)

    if isinstance(shap_output, list):
        shap_row = np.asarray(shap_output[0]).reshape(-1)

    else:
        shap_array = np.asarray(shap_output)

        if shap_array.ndim == 3:
            # Common multiclass / multi-output shape:
            # [samples, features, outputs]
            shap_row = shap_array[0, :, -1]

        elif shap_array.ndim == 2:
            shap_row = shap_array[0]

        else:
            shap_row = shap_array.reshape(-1)

    return shap_row


## Step 8 — Build a local SHAP explanation

This helper explains one transaction and returns the five features with the largest absolute SHAP contributions.


In [ ]:
#  Step 8: Local SHAP Explanation

def explain_transaction(transaction):
    
    # Explain one transaction using the finalized XGBoost model.

    # Returns a dictionary containing:
    # -? fraud probability
    # -> frozen threshold
    # -> risk level
    # -> SHAP contribution table
    # -> top five contributing features
    

    if not isinstance(transaction, pd.DataFrame):
        raise TypeError("Transaction must be a pandas DataFrame.")

    if transaction.shape[0] != 1:
        raise ValueError( "Transaction DataFrame must contain exactly one row." )

    missing_columns = [  column for column in feature_columns
        if column not in transaction.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    transaction = transaction[feature_columns].copy()

    if not all(pd.api.types.is_numeric_dtype(transaction[column])
        for column in feature_columns
    ):
        raise TypeError("All transaction features must be numeric.")

    if transaction.isnull().any().any():
        raise ValueError("Transaction contains missing values." )

    probability = float(xgb_model.predict_proba(transaction)[0, 1]
    )

    risk_level = ("HIGH" if probability >= FINAL_THRESHOLD else "LOW")

    shap_row = get_shap_row( explainer, transaction)

    if len(shap_row) != len(feature_columns):
        raise ValueError( f"SHAP feature mismatch: got {len(shap_row)} "
            f"SHAP values for {len(feature_columns)} features."
        )

    shap_table = pd.DataFrame({
        "feature": feature_columns,
        "feature_value": transaction.iloc[0].values,
        "shap_value": shap_row
    })

    shap_table["abs_shap"] = (shap_table["shap_value"].abs())

    shap_table=(shap_table.sort_values("abs_shap",ascending=False).reset_index(drop=True))

    return {
        "fraud_probability": probability,
        "fraud_percentage": probability * 100,
        "threshold": FINAL_THRESHOLD,
        "risk_level": risk_level,
        "is_fraud": bool(probability >= FINAL_THRESHOLD),
        "shap_table": shap_table,
        "top_features": shap_table.head(5)
    }


In [ ]:
#  Step 9: Find a High-Risk Fraud Example for Explanation


fraud_transactions = (df[df["Class"] == 1].drop(columns=["Class"]))

fraud_probabilities = (xgb_model.predict_proba(fraud_transactions)[:, 1])

highest_risk_position = int(np.argmax(fraud_probabilities))

highest_risk_transaction = (fraud_transactions.iloc[[highest_risk_position]])

highest_risk_original_index = (highest_risk_transaction.index[0])

highest_risk_probability = float(fraud_probabilities[highest_risk_position])

highest_risk_actual_class = int(df.loc[ highest_risk_original_index, "Class"])



print("Highest-risk fraud example selected.")

print( "Original DataFrame index:",highest_risk_original_index)

print( "Fraud probability:",f"{highest_risk_probability:.6f}")

print( "Actual class:", highest_risk_actual_class)


Highest-risk fraud example selected.
Original DataFrame index: 223618
Fraud probability: 0.999999
Actual class: 1


In [ ]:
# Step 10: Generate Local SHAP  Explanation

high_risk_explanation = explain_transaction(
    highest_risk_transaction
)

print("=" * 60)
print("HIGH-RISK TRANSACTION — SHAP EXPLANATION")
print("=" * 60)

print(f"Fraud Probability : " f"{high_risk_explanation['fraud_percentage']:.2f}%")

print(f"Decision Threshold: "f"{high_risk_explanation['threshold']:.2f}")

print(f"Risk Level        : "f"{high_risk_explanation['risk_level']}")

print(f"Fraud Flag        : "f"{high_risk_explanation['is_fraud']}")

print("\nTop Contributing Features:")

print(
    high_risk_explanation["top_features"][
        ["feature", "feature_value", "shap_value"]
    ].to_string(index=False)
)


HIGH-RISK TRANSACTION — SHAP EXPLANATION
Fraud Probability : 100.00%
Decision Threshold: 0.03
Risk Level        : HIGH
Fraud Flag        : True

Top Contributing Features:
feature  feature_value  shap_value
    V14      -8.893726    3.960416
    V10      -4.400930    1.404564
     V4       7.232058    1.344305
    V12      -5.737815    1.319820
     V3      -5.463811    0.897966


## Step 11 — Local SHAP Waterfall Plot

This plot shows how the strongest feature contributions move the model output for the selected transaction.


In [ ]:
# Step 11: SHAP Waterfall Plot

shap_table = high_risk_explanation["shap_table"]

shap_values_for_plot = shap_table.set_index("feature")["shap_value"]

# Recalculate SHAP values in the original
# feature order for the SHAP Explanation object.
original_order_shap = get_shap_row(
    explainer,
    highest_risk_transaction
)

base_value = explainer.expected_value

if isinstance(base_value, np.ndarray):
    base_value = float(np.asarray(base_value).reshape(-1)[-1])
else:
    base_value = float(base_value)

waterfall_explanation = shap.Explanation(
    values=original_order_shap,
    base_values=base_value,
    data=highest_risk_transaction.iloc[0].values,
    feature_names=feature_columns
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(
    waterfall_explanation,
    max_display=10,
    show=False
)

plt.tight_layout()
plt.savefig( "../shap_local_waterfall.png",dpi=300,bbox_inches="tight")

plt.show()
plt.savefig("shap_local_waterfall.png")


## Step 12 — Global SHAP Summary

For a compact project demonstration, explain a sample of transactions from the dataset rather than using the entire dataset.

The sample is drawn only for explainability visualization; this is not model training or threshold optimization.


In [ ]:
#Step 12: Global SHAP Summary

SHAP_SAMPLE_SIZE = min(1000, len(df))

shap_sample = (df.drop(columns=["Class"]).sample(n=SHAP_SAMPLE_SIZE,random_state=42))

shap_sample_values = explainer.shap_values(shap_sample)

# Normalize possible SHAP output shapes.
if isinstance(shap_sample_values, list):
    shap_matrix = np.asarray(shap_sample_values[0])

else:
    shap_array = np.asarray(shap_sample_values)

    if shap_array.ndim == 3:
        shap_matrix = shap_array[:, :, -1]
    else:
        shap_matrix = shap_array

print("SHAP sample shape:",shap_sample.shape)

shap.summary_plot(shap_matrix,shap_sample,show=False)

plt.tight_layout()
plt.savefig( "../shap_summary.png",dpi=300, bbox_inches="tight")

plt.show()
plt.savefig("shap_summary.png")


## Step 13 — Save Top Features for the High-Risk Example


In [ ]:
# Step 13: Save Top Features

top_features_output = (
    high_risk_explanation["top_features"][
        [
            "feature",
            "feature_value",
            "shap_value"
        ]
    ]
)

output_path = (Path("../top_features_high_risk_transaction.csv"))

top_features_output.to_csv(output_path,index=False)

print("Saved:", output_path.resolve())


Saved: C:\Users\HP\OneDrive\Desktop\Fraud Risk Detect\top_features_high_risk_transaction.csv


In [ ]:
#  Step 14: Final Explainability Check

print("=" * 60)
print("FINAL SHAP CONFIGURATION CHECK")
print("=" * 60)

print("Model: XGBoost")
print(f"Frozen decision threshold: "f"{FINAL_THRESHOLD:.2f}")
print("Final model artifact:",final_model_path.name)
print("Final config artifact:", final_config_path.name)

print("\nHigh-risk explanation generated successfully.")
print("SHAP waterfall: ../shap_local_waterfall.png")
print("SHAP summary  : ../shap_summary.png")
print( "Top features CSV: ../top_features_high_risk_transaction.csv")


FINAL SHAP CONFIGURATION CHECK
Model: XGBoost
Frozen decision threshold: 0.03
Final model artifact: final_xgb_model.pkl
Final config artifact: final_risk_config.pkl

High-risk explanation generated successfully.
SHAP waterfall: ../shap_local_waterfall.png
SHAP summary  : ../shap_summary.png
Top features CSV: ../top_features_high_risk_transaction.csv


## End of Step 7

This notebook now uses exactly the same finalized artifacts as Step 4:

```text
Step 6
  |─ final_xgb_model.pkl
  |─ final_risk_config.pkl
              |
        ┌─────┴─────┐
        |           |
     Step 4       Step 7
   Risk Engine     SHAP
        |           |
    Probability   Explanation
    Threshold     Top drivers
    Risk level    Waterfall
                  Summary
```

The final held-out test metrics remain the responsibility of Step 6.
